<a href="https://colab.research.google.com/github/dannydanny123405-gif/final_project/blob/main/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 좌석 추천 기능 업데이트

기존 Gradio 인터페이스에 '파울볼/홈런볼' 관련 질문을 추가하고, `recommend_seat` 함수를 업데이트하여 모든 사용자 선호도를 종합적으로 고려한 추천 결과를 제공합니다. 이제 '익사이팅존'과 '외야석'도 추천에 포함됩니다.

In [75]:
def _get_estimated_price(base_price_min, base_price_max):
    """
    좌석의 예상 가격을 계산합니다.
    """
    if not base_price_min or not base_price_max:
        return '가격 정보 없음'
    estimated_base_price = (base_price_min + base_price_max) / 2
    return f"{int(estimated_base_price):,}원"

def recommend_seat_updated(
    has_table: str,
    fan_type: str = None,
    want_cheer_active: str = None,
    is_with_child: str = None # New parameter
) -> str:
    prices_and_features = {
        '프리미엄석': {'base_min': 50000, 'base_max': 80000, 'feature': '내야 1층, 포수 후면 또는 덕아웃 부근에 위치하여 경기장 전체를 한눈에 조망하며 최상의 시야와 함께 테이블이 제공되어 편안하고 품격 있는 관람을 선사합니다. VIP 서비스가 제공될 수 있으며, 선수들의 움직임을 가장 가깝게 관찰할 수 있습니다.'},
        '테이블석': {'base_min': 40000, 'base_max': 70000, 'feature': '내야 각 구역(1루, 3루, 중앙)에 전략적으로 배치되어 있으며, 개별 테이블이 있어 음식과 음료를 편안하게 즐기며 여유로운 관람이 가능합니다. 주로 2층 이상의 높이에서 안정적인 시야를 제공합니다.'},
        '익사이팅존': {'base_min': 35000, 'base_max': 50000, 'feature': '그라운드와 가장 근접한 내야 구역으로, 펜스 바로 뒤에 위치하여 선수들의 생생한 움직임과 타구의 박진감을 만끽할 수 있습니다. 파울볼/홈런볼을 직접 잡을 가능성이 높아 스릴 넘치는 경험을 할 수 있습니다.'},
        '레드석': {'base_min': 30000, 'base_max': 55000, 'feature': '내야 1층과 2층 사이에 위치하며, 주로 응원단상 주변에 배치되어 있습니다. 응원단장, 치어리더와 함께 열정적인 응원을 펼치기에 최적화된 좌석입니다. 경기장 분위기를 주도하고 싶은 홈팀 팬에게 강력 추천합니다.'},
        '외야석': {'base_min': 15000, 'base_max': 25000, 'feature': '좌측 및 우측 외야에 위치하여 탁 트인 시야를 제공하며, 홈런볼을 잡을 기회가 있습니다. 비교적 자유로운 분위기에서 경기를 즐기실 수 있으며, 가족 단위 관람객에게 인기가 많습니다. 가격대가 합리적인 편입니다.'},
        '오렌지석_home_active': {'base_min': 25000, 'base_max': 45000, 'feature': '1루 응원단상과 가장 인접한 곳에 위치하여 홈팀의 활기찬 응원 문화를 즐기기에 이상적입니다. 열정적인 홈팀 팬들과 함께 호흡하며 응원할 수 있습니다. 선수들의 응원가를 따라 부르며 경기를 더 신나게 즐길 수 있습니다.'},
        '오렌지석_away_active': {'base_min': 25000, 'base_max': 45000, 'feature': '3루 응원단상과 가장 인접한 곳에 위치하여 원정팀의 활기찬 응원 문화를 즐기기에 이상적입니다. 열정적인 원정팀 팬들과 함께 호흡하며 응원할 수 있습니다. 원정팀 선수들을 위한 특별 응원도 경험할 수 있습니다.'},
        '블루석_home_general': {'base_min': 25000, 'base_max': 40000, 'feature': '내야 1루와 가까운 중앙 방향에 분포된 좌석입니다. 비교적 좋은 시야를 제공하며, 합리적인 가격으로 경기를 차분하게 관람하기 좋습니다. 홈팀 팬을 위한 일반적인 좌석입니다.'},
        '네이비석_home_general': {'base_min': 22000, 'base_max': 38000, 'feature': '내야 1루와 가까운 외야 방향에 분포된 좌석입니다. 가성비가 좋은 좌석으로, 편안하게 경기를 즐기기에 적합합니다. 홈팀 팬을 위한 일반적인 좌석입니다.'},
        '블루석_away_general': {'base_min': 25000, 'base_max': 40000, 'feature': '내야 3루와 가까운 중앙 방향에 분포된 좌석입니다. 비교적 좋은 시야를 제공하며, 합리적인 가격으로 경기를 차분하게 관람하기 좋습니다. 원정팀 팬을 위한 일반적인 좌석입니다.'},
        '네이비석_away_general': {'base_min': 22000, 'base_max': 38000, 'feature': '내야 3루와 가까운 외야 방향에 분포된 좌석입니다. 가성비가 좋은 좌석으로, 편안하게 경기를 즐기기에 적합합니다. 원정팀 팬을 위한 일반적인 좌석입니다.'},
        '블루석_neutral_general': {'base_min': 23000, 'base_max': 37000, 'feature': '내야 중앙에서 외야 방향으로 분포된 좌석 중 좋은 시야를 확보할 수 있는 구역입니다. 어느 팀을 응원하든 편안하게 경기를 즐길 수 있습니다.'},
        '네이비석_neutral_general': {'base_min': 20000, 'base_max': 35000, 'feature': '내야 중앙에서 외야 방향으로 넓게 분포된 좌석 중 가성비가 좋은 구역입니다. 합리적인 가격으로 경기를 즐기기에 좋습니다.'}
    }

    recommendations = []

    # 1. 테이블이 있는 좌석 선호 여부
    if has_table == '네, 선호해요':
        prem_price = _get_estimated_price(prices_and_features['프리미엄석']['base_min'], prices_and_features['프리미엄석']['base_max'])
        table_price = _get_estimated_price(prices_and_features['테이블석']['base_min'], prices_and_features['테이블석']['base_max'])
        return (
            f"**테이블이 있는 좌석을 선호하시는군요! 다음 좌석들을 추천드립니다:**\n\n"
            f"- **프리미엄석** (예상: {prem_price})\n  {prices_and_features['프리미엄석']['feature']}\n\n"
            f"- **테이블석** (예상: {table_price})\n  {prices_and_features['테이블석']['feature']}"
        )

    # 2. 응원 분위기 및 팬 유형
    if fan_type == '원정팀 팬입니다':
        if want_cheer_active == '네, 함께 열정적으로 응원하고 싶어요':
            if is_with_child == '네': # If user is with child
                orange_price = _get_estimated_price(prices_and_features['오렌지석_away_active']['base_min'], prices_and_features['오렌지석_away_active']['base_max'])
                red_price = _get_estimated_price(prices_and_features['레드석']['base_min'], prices_and_features['레드석']['base_max'])
                recommendations.append(
                    f"**아이와 함께 원정팀을 열정적으로 응원하고 싶으시면:**\n"
                    f"- **원정팀 응원석 (3루 오렌지석)** (예상: {orange_price})\n  {prices_and_features['오렌지석_away_active']['feature']}\n"
                    f"- **레드석** (예상: {red_price})\n  {prices_and_features['레드석']['feature']}"
                )
            elif is_with_child == '아니오': # If user is NOT with child
                exciting_price = _get_estimated_price(prices_and_features['익사이팅존']['base_min'], prices_and_features['익사이팅존']['base_max'])
                recommendations.append(
                    f"**아이 없이 원정팀을 열정적으로 응원하고 싶으시면:**\n"
                    f"- **익사이팅존** (예상: {exciting_price})\n  {prices_and_features['익사이팅존']['feature']}"
                )
            else: # Fallback if is_with_child is None
                orange_price = _get_estimated_price(prices_and_features['오렌지석_away_active']['base_min'], prices_and_features['오렌지석_away_active']['base_max'])
                red_price = _get_estimated_price(prices_and_features['레드석']['base_min'], prices_and_features['레드석']['base_max'])
                exciting_price = _get_estimated_price(prices_and_features['익사이팅존']['base_min'], prices_and_features['익사이팅존']['base_max'])
                recommendations.append(
                    f"**원정팀을 열정적으로 응원하고 싶으시면:**\n"
                    f"- **원정팀 응원석 (3루 오렌지석)** (예상: {orange_price})\n  {prices_and_features['오렌지석_away_active']['feature']}\n"
                    f"- **레드석** (예상: {red_price})\n  {prices_and_features['레드석']['feature']}\n"
                    f"- **익사이팅존** (예상: {exciting_price})\n  {prices_and_features['익사이팅존']['feature']}"
                )
        else: # 조용히 관람하고 싶어요
            blue_price = _get_estimated_price(prices_and_features['블루석_away_general']['base_min'], prices_and_features['블루석_away_general']['base_max'])
            navy_price = _get_estimated_price(prices_and_features['네이비석_away_general']['base_min'], prices_and_features['네이비석_away_general']['base_max'])
            recommendations.append(f"**원정팀 경기를 조용히 관람하고 싶으시면:**\n"
                                   f"- **원정팀 블루석** (예상: {blue_price})\n  {prices_and_features['블루석_away_general']['feature']}\n"
                                   f"- **원정팀 네이비석** (예상: {navy_price})\n  {prices_and_features['네이비석_away_general']['feature']}")
    elif fan_type == '홈팀 팬입니다':
        if want_cheer_active == '네, 함께 열정적으로 응원하고 싶어요':
            # NEW LOGIC FOR Q4: 아이와 함께 왔나요?
            if is_with_child == '네': # If user is with child
                orange_price = _get_estimated_price(prices_and_features['오렌지석_home_active']['base_min'], prices_and_features['오렌지석_home_active']['base_max'])
                red_price = _get_estimated_price(prices_and_features['레드석']['base_min'], prices_and_features['레드석']['base_max'])
                recommendations.append(
                    f"**아이와 함께 홈팀을 열정적으로 응원하고 싶으시면:**\n"
                    f"- **홈팀 응원석 (1루 오렌지석)** (예상: {orange_price})\n  {prices_and_features['오렌지석_home_active']['feature']}\n"
                    f"- **레드석** (예상: {red_price})\n  {prices_and_features['레드석']['feature']}"
                )
            elif is_with_child == '아니오': # If user is NOT with child
                exciting_price = _get_estimated_price(prices_and_features['익사이팅존']['base_min'], prices_and_features['익사이팅존']['base_max'])
                recommendations.append(
                    f"**아이 없이 홈팀을 열정적으로 응원하고 싶으시면:**\n"
                    f"- **익사이팅존** (예상: {exciting_price})\n  {prices_and_features['익사이팅존']['feature']}"
                )
            else: # Fallback if is_with_child is None (e.g., if Q4 was not visible or answered)
                  # In this case, recommend all three for passionate cheering as a general option.
                orange_price = _get_estimated_price(prices_and_features['오렌지석_home_active']['base_min'], prices_and_features['오렌지석_home_active']['base_max'])
                red_price = _get_estimated_price(prices_and_features['레드석']['base_min'], prices_and_features['레드석']['base_max'])
                exciting_price = _get_estimated_price(prices_and_features['익사이팅존']['base_min'], prices_and_features['익사이팅존']['base_max'])
                recommendations.append(
                    f"**홈팀을 열정적으로 응원하고 싶으시면:**\n"
                    f"- **홈팀 응원석 (1루 오렌지석)** (예상: {orange_price})\n  {prices_and_features['오렌지석_home_active']['feature']}\n"
                    f"- **레드석** (예상: {red_price})\n  {prices_and_features['레드석']['feature']}\n"
                    f"- **익사이팅존** (예상: {exciting_price})\n  {prices_and_features['익사이팅존']['feature']}"
                )
        else: # 조용히 관람하고 싶어요
            blue_price = _get_estimated_price(prices_and_features['블루석_home_general']['base_min'], prices_and_features['블루석_home_general']['base_max'])
            navy_price = _get_estimated_price(prices_and_features['네이비석_home_general']['base_min'], prices_and_features['네이비석_home_general']['base_max'])
            recommendations.append(f"**홈팀 경기를 조용히 관람하고 싶으시면:**\n"
                                   f"- **홈팀 블루석** (예상: {blue_price})\n  {prices_and_features['블루석_home_general']['feature']}\n"
                                   f"- **홈팀 네이비석** (예상: {navy_price})\n  {prices_and_features['네이비석_home_general']['feature']}")
    elif fan_type == '어느 팀 팬도 아니에요':
        blue_price = _get_estimated_price(prices_and_features['블루석_neutral_general']['base_min'], prices_and_features['블루석_neutral_general']['base_max'])
        navy_price = _get_estimated_price(prices_and_features['네이비석_neutral_general']['base_min'], prices_and_features['네이비석_neutral_general']['base_max'])
        recommendations.append(f"**어느 팀 팬도 아니시거나, 일반적인 관람을 원하시면:**\n"
                               f"- **블루석 (중립 관중)** (예상: {blue_price})\n  {prices_and_features['블루석_neutral_general']['feature']}\n"
                               f"- **네이비석 (중립 관중)** (예상: {navy_price})\n  {prices_and_features['네이비석_neutral_general']['feature']}")

    if not recommendations:
        # 최종 추천이 없을 경우 일반적인 외야석이나 블루/네이비석을 제안
        outfield_price = _get_estimated_price(prices_and_features['외야석']['base_min'], prices_and_features['외야석']['base_max'])
        blue_price = _get_estimated_price(prices_and_features['블루석_neutral_general']['base_min'], prices_and_features['블루석_neutral_general']['base_max'])
        navy_price = _get_estimated_price(prices_and_features['네이비석_neutral_general']['base_min'], prices_and_features['네이비석_neutral_general']['base_max'])

        recommendations.append(
            f"**선택하신 조건에 맞는 특별한 좌석은 없지만, 다음 좌석들을 고려해볼 수 있습니다:**\n"
            f"- **외야석** (예상: {outfield_price})\n  {prices_and_features['외야석']['feature']}\n"
            f"- **블루석** (예상: {blue_price})\n  {prices_and_features['블루석_neutral_general']['feature']}\n"
            f"- **네이비석** (예상: {navy_price})\n  {prices_and_features['네이비석_neutral_general']['feature']}"
        )

    result = "**고객님께 적합한 좌석을 추천드립니다:**\n\n" + "\n\n".join(recommendations)
    return result

In [1]:
import gradio as gr
import datetime

def get_essential_items_structured():
    return {
        'spring_autumn': {
            'label': '봄·가을 (3~5월, 9~10월)',
            'items': [
                '**두툼한 아우터**: 낮과 밤의 극심한 일교차와 저녁 찬바람에 대비합니다.',
                '**담요 및 휴대용 방석**: 플라스틱 의자에서 올라오는 하체 한기를 막아줍니다.',
                '**핫팩**: 7회 이후 급격히 쌀쌀해지는 야간 경기 필수 생존템입니다.',
                '**인공눈물 및 마스크**: 봄철 황사와 미세먼지, 야외 꽃가루를 차단합니다.'
            ]
        },
        'summer': {
            'label': '☀️ 한여름 (6~8월)',
            'items': [
                '**자외선 차단제, 선글라스, 모자**: 낮 경기와 해 질 무렵의 강렬한 직사광선을 막아줍니다.',
                '**휴대용 선풍기 및 쿨패치**: 가만히 있어도 땀이 흐르는 관중석 더위를 식혀줍니다.',
                '**얼린 생수**: 구장 안에서 음료가 금방 미지근해지는 것을 방지합니다.',
                '**벌레 기피제**: 야간 조명을 보고 몰려드는 모기와 벌레를 쫓아냅니다.'
            ]
        },
        'rainy': {
            'label': '🌧️ 비 오는 날 (우천 강행 경기)',
            'items': [
                '**우비**: 뒷사람의 시야를 방해하는 우산 대신 반드시 착용해야 합니다.',
                '**대형 쓰레기봉투**: 의자 아래 바닥 물기에 가방과 짐이 젖지 않게 통째로 넣습니다.',
                '**캡모자**: 우비 모자가 흘러내리지 않게 고정하고 얼굴로 치는 비를 막습니다.',
                '**스포츠 수건 및 여벌 옷**: 경기 관람 후 축축해진 몸과 옷을 정돈하고 귀가합니다.'
            ]
        },
        'common': {
            'label': '🎒 사계절 공통 필수품',
            'items': [
                '**보조배터리**: 모바일 티켓 확인과 응원 사진 촬영으로 인한 방전을 막습니다.',
                '**물티슈 및 휴지**: 먼지 쌓인 좌석은 야외 특성상 많고, 치킨이나 떡볶이 등 음식을 먹을 때 수시로 사용합니다.',
                '**크로스백**: 응원 도구를 흔들거나 박수를 칠 때 양손을 자유롭게 해줍니다.'
            ]
        }
    }

def filter_essential_items_by_date(selected_date_str):
    if not selected_date_str:
        return "날짜를 선택해 주세요."

    try:
        selected_date = datetime.datetime.strptime(selected_date_str, '%Y-%m-%d').date()
    except ValueError:
        return "날짜 형식이 올바르지 않습니다. YYYY-MM-DD 형식으로 입력해주세요."

    month = selected_date.month

    # Get game info to check for rain probability
    game_info = game_schedule.get(selected_date)
    rain_prob = game_info.get('rain_prob', 0) if game_info else 0 # Default to 0 if no game or no rain_prob

    all_items = get_essential_items_structured()
    seasonal_items_to_display = []

    # Add seasonal items based on month
    if 3 <= month <= 5 or 9 <= month <= 10:
        seasonal_items_to_display.append(all_items['spring_autumn'])
    elif 6 <= month <= 8:
        seasonal_items_to_display.append(all_items['summer'])

    # Conditionally include rainy day items based on rain probability
    if rain_prob >= 30:
        seasonal_items_to_display.append(all_items['rainy'])

    # Always include common items
    seasonal_items_to_display.append(all_items['common'])

    formatted_output = "### ⚾ 직관시 준비물 ⚾\n\n"
    if game_info:
        formatted_output += f"**선택하신 날짜의 강수확률: {rain_prob}%**\n\n"
    else:
        formatted_output += f"**선택하신 날짜의 강수확률 정보 없음**\n\n"

    for season_data in seasonal_items_to_display:
        formatted_output += f"### {season_data['label']}\n"
        for item in season_data['items']:
            formatted_output += f"*   {item}\n"
        formatted_output += "\n"
    return formatted_output

# Dummy Game Schedule for demonstration, including more KBO Jamsil teams for 2026 and rain probability
game_schedule = {
    datetime.date(2026, 6, 12): {'home': 'LG 트윈스', 'away': 'SSG 랜더스', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 10, 'max_temp': 28, 'min_temp': 20},
    datetime.date(2026, 6, 13): {'home': 'LG 트윈스', 'away': 'SSG 랜더스', 'time': '17:00', 'stadium': '잠실', 'rain_prob': 20, 'max_temp': 29, 'min_temp': 21},
    datetime.date(2026, 6, 14): {'home': 'LG 트윈스', 'away': 'SSG 랜더스', 'time': '14:00', 'stadium': '잠실', 'rain_prob': 50, 'max_temp': 25, 'min_temp': 19},
    datetime.date(2026, 6, 16): {'home': '두산 베어스', 'away': '키움 히어로즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 0, 'max_temp': 30, 'min_temp': 22},
    datetime.date(2026, 6, 17): {'home': '두산 베어스', 'away': '키움 히어로즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 35, 'max_temp': 27, 'min_temp': 20},
    datetime.date(2026, 6, 18): {'home': '두산 베어스', 'away': '키움 히어로즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 60, 'max_temp': 24, 'min_temp': 18},
    datetime.date(2026, 7, 1): {'home': 'LG 트윈스', 'away': 'NC 다이노스', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 70, 'max_temp': 26, 'min_temp': 22},
    datetime.date(2026, 7, 2): {'home': 'LG 트윈스', 'away': 'NC 다이노스', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 80, 'max_temp': 25, 'min_temp': 21},
    datetime.date(2026, 7, 3): {'home': 'LG 트윈스', 'away': 'NC 다이노스', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 90, 'max_temp': 24, 'min_temp': 20},
    datetime.date(2026, 7, 10): {'home': '두산 베어스', 'away': '롯데 자이언츠', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 15, 'max_temp': 31, 'min_temp': 23},
    datetime.date(2026, 7, 11): {'home': '두산 베어스', 'away': '롯데 자이언츠', 'time': '17:00', 'stadium': '잠실', 'rain_prob': 25, 'max_temp': 32, 'min_temp': 24},
    datetime.date(2026, 7, 12): {'home': '두산 베어스', 'away': '롯데 자이언츠', 'time': '14:00', 'stadium': '잠실', 'rain_prob': 30, 'max_temp': 29, 'min_temp': 22},
    datetime.date(2026, 8, 5): {'home': 'LG 트윈스', 'away': '한화 이글스', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 5, 'max_temp': 33, 'min_temp': 25},
    datetime.date(2026, 8, 6): {'home': 'LG 트윈스', 'away': '한화 이글스', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 10, 'max_temp': 34, 'min_temp': 26},
    datetime.date(2026, 8, 7): {'home': 'LG 트윈스', 'away': '한화 이글스', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 20, 'max_temp': 32, 'min_temp': 24},
    datetime.date(2026, 8, 20): {'home': '두산 베어스', 'away': 'KIA 타이거즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 40, 'max_temp': 28, 'min_temp': 22},
    datetime.date(2026, 8, 21): {'home': '두산 베어스', 'away': 'KIA 타이거즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 45, 'max_temp': 27, 'min_temp': 21},
    datetime.date(2026, 8, 22): {'home': '두산 베어스', 'away': 'KIA 타이거즈', 'time': '17:00', 'stadium': '잠실', 'rain_prob': 55, 'max_temp': 26, 'min_temp': 20},
    datetime.date(2026, 9, 2): {'home': 'LG 트윈스', 'away': 'KT 위즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 10, 'max_temp': 27, 'min_temp': 19},
    datetime.date(2026, 9, 3): {'home': 'LG 트윈스', 'away': 'KT 위즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 20, 'max_temp': 26, 'min_temp': 18},
    datetime.date(2026, 9, 4): {'home': 'LG 트윈스', 'away': 'KT 위즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 25, 'max_temp': 25, 'min_temp': 17},
    datetime.date(2026, 9, 15): {'home': '두산 베어스', 'away': '삼성 라이온즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 30, 'max_temp': 23, 'min_temp': 15},
    datetime.date(2026, 9, 16): {'home': '두산 베어스', 'away': '삼성 라이온즈', 'time': '18:30', 'stadium': '잠실', 'rain_prob': 40, 'max_temp': 22, 'min_temp': 14}
}

def get_game_info(selected_date_str):
    if not selected_date_str:
        return "날짜를 선택해 주세요."

    try:
        selected_date = datetime.datetime.strptime(selected_date_str, '%Y-%m-%d').date()
    except ValueError:
        return "날짜 형식이 올바르지 않습니다. YYYY-MM-DD 형식으로 입력해주세요."

    game = game_schedule.get(selected_date)
    if game and game['stadium'] == '잠실': # Check if the game is at Jamsil
        rain_prob = game.get('rain_prob', '정보 없음')
        max_temp = game.get('max_temp', '정보 없음')
        min_temp = game.get('min_temp', '정보 없음')

        cancellation_msg = ""
        if isinstance(rain_prob, (int, float)):
            if rain_prob >= 70:
                cancellation_msg = " (우천 취소 가능성이 매우 높습니다.)"
            elif rain_prob >= 40:
                cancellation_msg = " (우천 취소 가능성이 있습니다.)"

        return (
            f"**{selected_date.strftime('%Y년 %m월 %d일')} 잠실 경기 정보:**\n\n"
            f"*   **홈팀:** {game['home']}\n"
            f"*   **원정팀:** {game['away']}\n"
            f"*   **경기 시간:** {game['time']}\n"
            f"*   **강수확률:** {rain_prob}%{cancellation_msg}\n"
            f"*   **최고 기온:** {max_temp}°C\n"
            f"*   **최저 기온:** {min_temp}°C"
        )
    else:
        return f"{selected_date.strftime('%Y년 %m월 %d일')}에는 잠실구장에 예정된 경기가 없습니다."

def submit_feedback(feedback_text):
    if feedback_text:
        print(f"User feedback received: {feedback_text}")
        return "피드백을 보내주셔서 감사합니다!"
    else:
        return "피드백 내용을 입력해주세요."

with gr.Blocks() as demo_updated:
    gr.Markdown("# ⚾ 잠실 야구장 좌석 가이드: 당신의 완벽한 직관을 위한 선택")

    q1 = gr.Radio(['네, 선호해요', '아니오, 괜찮습니다'], label='1. 편안한 관람을 위해 테이블이 있는 좌석을 선호하시나요?', value='아니오, 괜찮습니다')

    with gr.Column(visible=True) as extra_ui_group:
        q2 = gr.Radio(['원정팀 팬입니다', '홈팀 팬입니다', '어느 팀 팬도 아니에요'], label='2. 응원하시는 팀은 어디인가요?', value='홈팀 팬입니다')
        q3 = gr.Radio(['네, 함께 열정적으로 응원하고 싶어요', '아니오, 조용히 경기를 관람하고 싶어요'], label='3. 경기의 열기를 만끽하며 열정적인 응원을 원하시나요?', value='네, 함께 열정적으로 응원하고 싶어요')
        q4 = gr.Radio(['네', '아니오'], label='4. 아이와 함께 오셨나요?', visible=False)

    btn_recommend = gr.Button("나에게 맞는 좌석 추천받기", variant="primary")
    output_recommend = gr.Textbox(label="추천 결과", lines=20)

    # Date Input and Item components
    gr.Markdown("## 📅 직관 날짜 선택 및 필수 준비물")
    date_picker = gr.Textbox(label="직관 예정일 선택 (YYYY-MM-DD)", value=datetime.date.today().strftime('%Y-%m-%d'))
    output_game_info = gr.Markdown(label="경기 정보")
    output_essential_items_filtered = gr.Markdown(label="필수 준비물 (선택 날짜 기준)")

    gr.Markdown("## 📝 추천 결과 피드백")
    feedback_textbox = gr.Textbox(label="추천 결과에 대한 의견을 남겨주세요", placeholder="예: 추천이 도움이 되었습니다 / 다른 좌석도 고려하고 싶어요 등")
    feedback_button = gr.Button("피드백 제출하기")
    feedback_output = gr.Textbox(label="", interactive=False)

    q1.change(lambda x: gr.update(visible=(x == '아니오, 괜찮습니다')), q1, extra_ui_group)

    def update_q4_visibility(q2_val, q3_val):
        return gr.update(visible=((q2_val == '홈팀 팬입니다' or q2_val == '원정팀 팬입니다') and q3_val == '네, 함께 열정적으로 응원하고 싶어요'))

    q2.change(update_q4_visibility, [q2, q3], q4)
    q3.change(update_q4_visibility, [q2, q3], q4)

    btn_recommend.click(
        recommend_seat_updated,
        [q1, q2, q3, q4],
        output_recommend
    )

    # Event handler for date_picker
    date_picker.change(get_game_info, date_picker, output_game_info)
    date_picker.change(filter_essential_items_by_date, date_picker, output_essential_items_filtered)

    # Event handler for feedback
    feedback_button.click(submit_feedback, feedback_textbox, feedback_output)

demo_updated.launch(debug=True, share=True)

NameError: name 'recommend_seat_updated' is not defined

### 🛠️ GitHub 연동을 위한 준비

GitHub에 노트북을 등록하기 위해 필요한 도구들을 설치하고 Git 설정을 진행합니다. `nbdime`은 노트북 변경 사항을 더 쉽게 확인하기 위한 도구입니다.